# Caso 2: Pérdida de agua al hacer ejercicios

Este cuaderno muestra un ejemplo simple de **Naive Bayes** aplicado a un problema de clasificación.

## Contexto
Un gimnasio en Santiago quiere predecir el nivel de **pérdida de agua** de una persona durante una sesión de ejercicio, considerando variables como:

- duración del ejercicio
- temperatura del ambiente
- intensidad del ejercicio
- grupo etario

El objetivo es construir un modelo que clasifique si la pérdida de agua será **Baja**, **Media** o **Alta**.

## 1. Importar librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, ConfusionMatrixDisplay


## 2. Crear el set de datos

Cada fila representa una persona con ciertas características y el nivel de pérdida de agua observado.

In [ ]:
data = [
    [20, 'Fria',     'Baja',  'Joven',  'Baja'],
    [30, 'Templada', 'Media', 'Adulto', 'Media'],
    [45, 'Calurosa', 'Alta',  'Joven',  'Alta'],
    [25, 'Fria',     'Media', 'Adulto', 'Baja'],
    [50, 'Calurosa', 'Alta',  'Adulto', 'Alta'],
    [35, 'Templada', 'Alta',  'Joven',  'Media'],
    [40, 'Calurosa', 'Media', 'Mayor',  'Alta'],
    [15, 'Fria',     'Baja',  'Mayor',  'Baja'],
    [55, 'Calurosa', 'Alta',  'Joven',  'Alta'],
    [28, 'Templada', 'Media', 'Joven',  'Media'],
    [22, 'Fria',     'Baja',  'Adulto', 'Baja'],
    [38, 'Templada', 'Alta',  'Mayor',  'Media'],
    [60, 'Calurosa', 'Alta',  'Adulto', 'Alta'],
    [32, 'Templada', 'Media', 'Mayor',  'Media'],
    [18, 'Fria',     'Baja',  'Joven',  'Baja']
]

df = pd.DataFrame(data, columns=['duracion_min', 'temperatura', 'intensidad', 'edad_grupo', 'perdida_agua'])
df

## 3. Revisar el dataset

In [ ]:
print('Cantidad de registros:', df.shape[0])
print('Cantidad de columnas:', df.shape[1])
print('\nTipos de datos:')
print(df.dtypes)
print('\nDistribución de la variable objetivo:')
print(df['perdida_agua'].value_counts())

## 4. Gráfico 1: Distribución del nivel de pérdida de agua

In [ ]:
conteo_perdida = df['perdida_agua'].value_counts()

plt.figure(figsize=(7,5))
plt.bar(conteo_perdida.index, conteo_perdida.values)
plt.title('Cantidad de personas por nivel de pérdida de agua')
plt.xlabel('Nivel de pérdida de agua')
plt.ylabel('Cantidad')
plt.show()

## 5. Gráfico 2: Duración promedio según nivel de pérdida de agua

In [ ]:
promedios = df.groupby('perdida_agua')['duracion_min'].mean()

plt.figure(figsize=(7,5))
plt.bar(promedios.index, promedios.values)
plt.title('Duración promedio según nivel de pérdida de agua')
plt.xlabel('Nivel de pérdida de agua')
plt.ylabel('Duración promedio (minutos)')
plt.show()

## 6. Codificar variables categóricas

Naive Bayes necesita trabajar con valores numéricos, por eso transformamos las columnas categóricas.

In [ ]:
encoders = {}

for col in ['temperatura', 'intensidad', 'edad_grupo', 'perdida_agua']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

df

## 7. Separar variables predictoras y variable objetivo

In [ ]:
X = df.drop('perdida_agua', axis=1)
y = df['perdida_agua']

print('X:')
print(X.head())
print('\ny:')
print(y.head())

## 8. Dividir en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print('Tamaño entrenamiento:', X_train.shape)
print('Tamaño prueba:', X_test.shape)
print('Distribución y_train:')
print(y_train.value_counts().sort_index())
print('\nDistribución y_test:')
print(y_test.value_counts().sort_index())

## 9. Entrenar el modelo Naive Bayes

In [ ]:
modelo = GaussianNB()
modelo.fit(X_train, y_train)

## 10. Realizar predicciones

In [ ]:
y_pred = modelo.predict(X_test)

print('Valores reales:     ', list(y_test))
print('Valores predichos:  ', list(y_pred))

## 11. Evaluar el modelo

In [ ]:
etiquetas = sorted(y.unique())
nombres_clases = encoders['perdida_agua'].inverse_transform(etiquetas)

matriz = confusion_matrix(y_test, y_pred, labels=etiquetas)
accuracy = accuracy_score(y_test, y_pred)

print('Matriz de confusión:')
print(matriz)
print('\nAccuracy del modelo:', round(accuracy, 4))
print('\nReporte de clasificación:')
print(classification_report(
    y_test,
    y_pred,
    labels=etiquetas,
    target_names=nombres_clases,
    zero_division=0
))

## 12. Gráfico 3: Matriz de confusión

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=nombres_clases)
disp.plot()
plt.title('Matriz de confusión del modelo')
plt.show()

## 13. Probar con un nuevo caso

Caso nuevo:

- duración = 42 minutos
- temperatura = Calurosa
- intensidad = Alta
- grupo etario = Adulto

In [ ]:
nuevo_caso = pd.DataFrame([[42, 'Calurosa', 'Alta', 'Adulto']], columns=['duracion_min', 'temperatura', 'intensidad', 'edad_grupo'])
nuevo_caso

In [ ]:
nuevo_caso_codificado = nuevo_caso.copy()

for col in ['temperatura', 'intensidad', 'edad_grupo']:
    nuevo_caso_codificado[col] = encoders[col].transform(nuevo_caso_codificado[col])

nuevo_caso_codificado

In [ ]:
prediccion = modelo.predict(nuevo_caso_codificado)
resultado = encoders['perdida_agua'].inverse_transform(prediccion)

print('Predicción del modelo:', resultado[0])

## 14. Conclusión

En este caso, Naive Bayes permite estimar el nivel de **pérdida de agua** durante el ejercicio a partir de variables simples.

El cuaderno incluye gráficos para que los estudiantes puedan:

1. visualizar la distribución de clases
2. comparar la duración promedio según pérdida de agua
3. interpretar visualmente la matriz de confusión